In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_auc_score,
                             roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
import subprocess
subprocess.run(['pip', 'install', 'imbalanced-learn'])

CompletedProcess(args=['pip', 'install', 'imbalanced-learn'], returncode=0)

In [ ]:
df = pd.read_csv('C:/Users/durga/OneDrive/Desktop/titanic-project/creditcard.csv')
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

In [ ]:
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

In [ ]:
print("Class Distribution:")
print(df['Class'].value_counts())
print("\nFraud Percentage:",
      round(df['Class'].value_counts()[1] / len(df) * 100, 3), "%")

# Visualize
sns.countplot(x='Class', data=df,
              palette=['#2ecc71','#e74c3c'])
plt.title('Class Distribution (0=Genuine, 1=Fraud)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks([0,1], ['Genuine','Fraud'])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genuine
axes[0].hist(df[df['Class']==0]['Amount'],
             bins=50, color='#2ecc71', edgecolor='black')
axes[0].set_title('Genuine Transaction Amounts')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Count')

# Fraud
axes[1].hist(df[df['Class']==1]['Amount'],
             bins=50, color='#e74c3c', edgecolor='black')
axes[1].set_title('Fraud Transaction Amounts')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

print("Average Genuine Amount:", round(df[df['Class']==0]['Amount'].mean(), 2))
print("Average Fraud Amount:", round(df[df['Class']==1]['Amount'].mean(), 2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Class']==0]['Time'],
             bins=50, color='#3498db', edgecolor='black')
axes[0].set_title('Genuine Transaction Time')
axes[0].set_xlabel('Time (seconds)')

axes[1].hist(df[df['Class']==1]['Time'],
             bins=50, color='#e74c3c', edgecolor='black')
axes[1].set_title('Fraud Transaction Time')
axes[1].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 10))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', fmt='.1f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
scaler = StandardScaler()

df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

df.drop(columns=['Amount', 'Time'], inplace=True)

print("Scaling done!")
df.head()

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts())

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_sm).value_counts())

# Visualize
sns.countplot(x=y_train_sm, palette=['#2ecc71','#e74c3c'])
plt.title('Class Distribution After SMOTE')
plt.xticks([0,1], ['Genuine','Fraud'])
plt.show()

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr)*100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr,
      target_names=['Genuine','Fraud']))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf)*100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf,
      target_names=['Genuine','Fraud']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Genuine','Fraud'],
            yticklabels=['Genuine','Fraud'], ax=axes[0])
axes[0].set_title('Logistic Regression\nConfusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Genuine','Fraud'],
            yticklabels=['Genuine','Fraud'], ax=axes[1])
axes[1].set_title('Random Forest\nConfusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={round(roc_auc_score(y_test, y_prob_lr),3)})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={round(roc_auc_score(y_test, y_prob_rf),3)})')
plt.plot([0,1],[0,1],'k--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp.sort_values(ascending=False).head(10).plot(
    kind='bar', color='coral', figsize=(10, 5))
plt.title('Top 10 Important Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.show()

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_lr)*100, 2),
        round(accuracy_score(y_test, y_pred_rf)*100, 2)
    ],
    'ROC-AUC': [
        round(roc_auc_score(y_test, y_prob_lr), 3),
        round(roc_auc_score(y_test, y_prob_rf), 3)
    ]
})

print(results)
results.plot(x='Model', kind='bar', figsize=(8,5), colormap='Set2')
plt.title('Model Comparison')
plt.xticks(rotation=0)
plt.ylabel('Score')
plt.show()

## Conclusion

- **Dataset:** Credit Card Transactions (highly imbalanced)
- **Class Imbalance Handled:** SMOTE oversampling
- **Models Used:** Logistic Regression & Random Forest

### Key Findings:
- Only ~0.17% of transactions are fraudulent (severe imbalance)
- SMOTE helped balance the classes for better training
- Random Forest outperformed Logistic Regression
- High ROC-AUC score shows excellent fraud detection ability

### Important Metrics for Fraud Detection:
- **Recall** is most important — catch as many frauds as possible
- **Precision** — avoid flagging genuine transactions as fraud
- **F1-Score** — balance between precision and recall